# M05B: Context Window Strategies

Conversations grow with every turn — without management, costs multiply and you hit token limits.

**Topics:**
- Sliding window strategy
- Conversation summarization
- Choosing the right strategy

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🔀 Why Manage Context Locally?

In M04A we used `previous_response_id` to let OpenAI track conversations server-side — simple, but you can't trim or cap the growing context.  

Local management gives you full control over what the model sees and what you pay for.

---

## 🪟 Part 1: Sliding Window Strategy

Keep only the N most recent messages — oldest get dropped as new ones arrive.

In [ ]:
class ContextWindow:
    """Manage context with sliding window."""
    
    def __init__(self, max_turns=5):
        self.max_turns = max_turns
        self.message_history = []
    
    def add_message(self, speaker, text):
        """Add message and maintain window."""
        # speaker: 'user' or 'assistant'
        self.message_history.append({'speaker': speaker, 'text': text})
        
        # Keep only last N turns (N turns = 2N messages)
        max_messages = self.max_turns * 2
        if len(self.message_history) > max_messages:
            self.message_history[:] = self.message_history[-max_messages:]
    
    def get_messages(self):
        """Get current context."""
        return self.message_history.copy()
    
    def build_context(self):
        """Build context string for API calls."""
        context = ""
        for msg in self.message_history:
            context += f"{msg['speaker']}: {msg['text']}\n"
        return context


# --------------------------------------------------------------
print("✅ ContextWindow ready")

### Demo: Sliding Window in Action

First, we test with simulated data to see the sliding window trim messages.  

Then we connect it to the live API.

In [ ]:
print("🪟 SLIDING WINDOW DEMO")
print("="*60)

window = ContextWindow(max_turns=3)

# Add 5 turns (10 messages total) - watch oldest drop off
for i in range(1, 6):
    window.add_message("user", f"Question {i}")
    window.add_message("assistant", f"Answer {i}")
    message_count = len(window.get_messages())
    print(f"After turn {i}: {message_count} messages (max: 6)")

print(f"\n✅ Final context contains only last 3 turns (6 messages):")
for message in window.get_messages():
    print(f"  {message['speaker']:10} → {message['text']}")

print("="*60)

### Sliding Window with Live API

Now let's use the `ContextWindow` class in a real conversation and watch the window manage context automatically.

In [ ]:
print("🪟 SLIDING WINDOW + LIVE API")
print("="*60)

window = ContextWindow(max_turns=3)

questions = [
    "What is ML?",
    "Common algorithms?",
    "How to start?",
    "Best language?"
]

for turn_number, question in enumerate(questions, 1):
    window.add_message("user", question)
    
    response = client.responses.create(
        model=MODEL,
        input=window.build_context(),
        instructions="1-2 sentences. Be concise."
    )
    
    answer = response.output_text
    window.add_message("assistant", answer)
    
    print(f"\nTurn {turn_number}: {question}")
    print(f"Response: {answer[:80]}...")
    print(f"📊 {len(window.get_messages())} messages in window (max: 6)")

print("\n" + "="*60)
print(f"✅ Context capped at {window.max_turns} turns — older messages dropped automatically")
print("="*60)

---

## 📝 Part 2: Conversation Summarization

Instead of dropping old messages, compress them into a summary.  

Recent messages stay in full, older ones become a summary, originals get dropped.

### How Summarization Works

In [ ]:
# Manual summarization — what the class below automates
old_messages = (
    "user: What is machine learning?\n"
    "assistant: ML is a subset of AI that enables systems to learn from data.\n"
    "user: What are common algorithms?\n"
    "assistant: Common algorithms include linear regression, decision trees, and neural networks.\n"
    "user: Which is best for beginners?\n"
    "assistant: Linear regression is the best starting point — it's simple, interpretable, and teaches core concepts.\n"
    "user: What about deep learning?\n"
    "assistant: Deep learning uses neural networks with many layers. It excels at images, text, and speech but requires more data and compute.\n"
    "user: Do I need a GPU?\n"
    "assistant: For basic ML, no. For deep learning training, a GPU significantly speeds things up. Cloud GPUs are a good starting option."
)

print("📝 MANUAL SUMMARIZATION DEMO")
print("="*60)

response = client.responses.create(
    model=MODEL,
    input=f"Summarize this conversation:\n{old_messages}",
    instructions="1-2 sentences. Capture main topics discussed."
)

summary = response.output_text.strip()
print(f"Original: {len(old_messages)} chars")
print(f"Summary:  {len(summary)} chars")
print(f"Compression: {100 - (len(summary) * 100 // len(old_messages))}% reduction\n")
print(f"{summary}")
print("="*60)

### Automating Summarization

In [ ]:
class SummarizingContext:
    """Context manager with automatic summarization."""
    
    def __init__(self, client, summarize_after_turns=4, model=MODEL):
        self.client = client
        self.summarize_after_turns = summarize_after_turns
        self.model = model
        self.message_history = []
        self.summary = None
    
    def add_message(self, speaker, text):
        """Add message and trigger summarization if threshold reached."""
        # speaker: 'user' or 'assistant'
        self.message_history.append({'speaker': speaker, 'text': text})
        
        if len(self.message_history) >= self.summarize_after_turns * 2:
            self._summarize()
    
    def _summarize(self):
        """Compress old messages into a summary, keep recent ones."""
        # Oldest messages get summarized, newest 4 (2 turns) stay in full
        messages_to_summarize = self.message_history[:-4]
        
        # Build text from old messages
        conversation_text = ""
        for message in messages_to_summarize:
            conversation_text += f"{message['speaker']}: {message['text']}\n"
        
        summary_prompt = f"""Update the running summary with new conversation.

Existing summary:
{self.summary or "None yet."}

New conversation:
{conversation_text}

Return an updated summary (2-3 sentences). Preserve key facts and decisions."""
        
        # Note: In production, wrap this in try/except for robust error handling
        summary_response = self.client.responses.create(
            model=self.model,
            input=summary_prompt,
            instructions="Be concise. Preserve key facts. Merge old and new context."
        )
        self.summary = summary_response.output_text.strip()
        
        # Keep only the 4 most recent messages in place
        self.message_history[:] = self.message_history[-4:]
    
    def get_context(self):
        """Get current context: summary (if exists) + recent messages."""
        context = []
        
        # Prepend summary of older messages (if summarization has occurred)
        if self.summary:
            context.append({
                'speaker': 'summary',
                'text': f'Previous conversation summary: {self.summary}'
            })
        
        # Append recent messages (kept in full for accuracy)
        context.extend(self.message_history)
        return context


# --------------------------------------------------------------
print("✅ SummarizingContext ready")

### Demo: Summarization in Action

In [ ]:
print("📝 SUMMARIZATION DEMO")
print("="*60)

summarizer = SummarizingContext(client, summarize_after_turns=4)

conversation = [
    ("user", "What is machine learning?"),
    ("assistant", "Machine learning is a subset of AI that enables systems to learn from data."),
    ("user", "What are common algorithms?"),
    ("assistant", "Common algorithms include linear regression, decision trees, and neural networks."),
    ("user", "Which is best for beginners?"),
    ("assistant", "Linear regression is best for beginners - simple yet powerful."),
    ("user", "How do I implement it?"),
    ("assistant", "Use scikit-learn: from sklearn.linear_model import LinearRegression")
]

# Add first 3 turns (no summarization yet)
for speaker, text in conversation[:6]:
    summarizer.add_message(speaker, text)

print(f"After 3 turns: {len(summarizer.message_history)} messages in memory")
print(f"Summary: {summarizer.summary}\n")

# Add turn 4 (triggers summarization)
for speaker, text in conversation[6:]:
    summarizer.add_message(speaker, text)

print(f"After 4 turns: {len(summarizer.message_history)} messages in memory")
print(f"Summary: {summarizer.summary}\n")

# Show current context
print("Current context (summary + recent messages):")
print("-"*60)
for msg in summarizer.get_context():
    if msg['speaker'] == 'summary':
        print(f"  [summary]   {msg['text']}")
    else:
        print(f"  [{msg['speaker']:9}]   {msg['text']}")

print("="*60)

### 🔑 When to Use `previous_response_id` 

**`previous_response_id`:** Simple, but costs grow unchecked — best for short conversations.

**Local context management:** Full control over cost and content — best for long conversations where you need sliding windows or summarization.

---

### 💪 Your Turn: Budget-Aware Chatbot

Build a production chatbot that enforces token or cost budgets:
- Set a token budget (e.g., 1,000 tokens)
- Warn user at 80% usage
- Block new requests at 100% budget

**Challenge extensions:** Switch to dollar budget, add auto-recovery via summarization, or implement tiered budgets (free vs pro).

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Budget-Aware Chatbot
# --------------------------------------------------------------
# Objective: Combine ContextWindow with a token or cost budget.
#
# Use TokenCounter and CostTracker from M05A to add budget
# enforcement on top of the sliding window.

class BudgetChatbot:
    """Chatbot with sliding window and token budget enforcement."""
    
    def __init__(self, client, token_budget=1000, max_turns=5, model=MODEL):
        # TODO: Store client, budget, model
        # TODO: Initialize ContextWindow (from this notebook)
        # TODO: Initialize TokenCounter (from M05A) and tracking variables
        pass
    
    def chat(self, user_message):
        # TODO: Count input tokens
        # TODO: Check if exceeds budget → return error
        # TODO: Warn at 80% usage
        # TODO: Build context from window, make API call, track tokens
        pass
    
    def get_budget_status(self):
        # TODO: Return usage stats (used, remaining, percent)
        pass


# --------------------------------------------------------------
# TODO: Test your implementation
# chatbot = BudgetChatbot(client, token_budget=500)
# response = chatbot.chat("Explain machine learning")
# print(chatbot.get_budget_status())

print("💡 Build your budget-aware chatbot!")

---

## 🎯 Key Takeaways

**🪟 Sliding Window:**
- Simple, predictable, loses old context
- Use for chat, quick support, casual conversation

**📝 Summarization:**
- Preserves history, extra API cost
- Use for long technical conversations, research, tutoring

**🔑 Local vs Server-Side Context:**
- `previous_response_id` is simple but costs grow unchecked
- Local context management gives you full control over what the model sees
- Use local management for long conversations

**The Flow:** Choose strategy based on conversation type → Build context locally → Send only what the model needs

---

### 📍 Next Step

**M05C: Response Caching** — Cache API responses to eliminate duplicate calls and reduce costs.

---

## 🔧 Troubleshooting

**Sliding window not trimming messages?**
- Verify calculation: `max_messages = max_turns * 2`
- Check list slicing: `message_history[-max_messages:]` keeps newest
- Test with `max_turns=2` to see trimming behavior clearly

**Summarization not triggering?**
- Verify `len(message_history) >= summarize_after_turns * 2` threshold
- Check that `_summarize()` is being called (add debug prints)
- Ensure API call succeeds—catch exceptions

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---